In [4]:
import os
import io
import pandas as pd

# Input / output paths
input_file = "data_april_to_november.csv"
output_folder = "monthly_csv_exports"
os.makedirs(output_folder, exist_ok=True)

# ── NEW: normalize all lines to the header's column count ──
with open(input_file, "r", encoding="utf-8") as f:
    lines = f.read().splitlines()

lines = [line.rstrip(",") for line in lines]          # strip trailing commas
n_cols = len(lines[0].split(","))                     # count columns from header
lines = [",".join(line.split(",")[:n_cols]) for line in lines]  # truncate each row

df = pd.read_csv(io.StringIO("\n".join(lines)))

# Set your date column name here if known
date_col = "date"

# If "date" is not present, auto-detect a usable datetime column
if date_col not in df.columns:
    for col in df.columns:
        parsed = pd.to_datetime(df[col], errors="coerce")
        if parsed.notna().sum() > 0:
            date_col = col
            break
    else:
        raise ValueError("No datetime-like column found in the CSV.")

# Convert to datetime
df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df = df.dropna(subset=[date_col])

# Create one CSV per month
for month, group in df.groupby(df[date_col].dt.to_period("M")):
    out_file = os.path.join(output_folder, f"{month}.csv")
    group.to_csv(out_file, index=False)

print(f"Done. Monthly CSV files saved in: {output_folder}")

C:\Users\clemr\AppData\Local\Temp\ipykernel_19364\2271358261.py:38: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  for month, group in df.groupby(df[date_col].dt.to_period("M")):


Done. Monthly CSV files saved in: monthly_csv_exports
